# 04 - U-Net Segmentation

Notebook huấn luyện và kiểm tra mô hình U-Net để phân đoạn vùng tổn thương da trong ảnh ISIC.

Pipeline: Image → Preprocessing → U-Net → Probability Mask → Binary Mask.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF

## 1. Configuration

In [ ]:
IMAGE_DIR = Path("../data/images/train")
MASK_DIR = Path("../data/masks/train")
RESULT_DIR = Path("../results/segmentation")

RESULT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (256, 256)
NUM_EPOCHS = 5
BATCH_SIZE = 4
LEARNING_RATE = 0.001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Image directory:", IMAGE_DIR.resolve())
print("Mask directory :", MASK_DIR.resolve())
print("Image size     :", IMAGE_SIZE)
print("Device         :", DEVICE)

## 2. Find Image-Mask Pairs

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

image_files = sorted([
    file for file in IMAGE_DIR.rglob("*")
    if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
])

pairs = []

for image_path in image_files:
    mask_path = MASK_DIR / f"{image_path.stem}.png"

    if mask_path.exists():
        pairs.append((image_path, mask_path))

print("Images found:", len(image_files))
print("Valid pairs :", len(pairs))

## 3. Dataset Class

Ảnh được resize về 256×256 và chuyển thành Tensor.

Mask được resize bằng Nearest Neighbor và chuyển thành Binary Mask.

In [ ]:
class ISICSegmentationDataset(Dataset):
    def __init__(self, pairs, image_size=(256, 256)):
        self.pairs = pairs
        self.image_size = image_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = image.resize(
            self.image_size,
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            self.image_size,
            Image.Resampling.NEAREST
        )

        image = TF.to_tensor(image)

        mask = np.array(mask)
        mask = (mask > 0).astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


dataset = ISICSegmentationDataset(
    pairs,
    image_size=IMAGE_SIZE
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Dataset size:", len(dataset))

## 4. Visualize Training Sample

In [ ]:
if len(dataset) > 0:
    image, mask = dataset[0]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(image.permute(1, 2, 0))
    axes[0].set_title("Input Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze(0), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy dữ liệu.")

## 5. U-Net Model

U-Net gồm Encoder, Bottleneck và Decoder.

Skip Connections giúp truyền thông tin chi tiết từ Encoder sang Decoder.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.output = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


model = UNet().to(DEVICE)

print(model)

## 6. Dice Loss

In [ ]:
def dice_loss(logits, targets, smooth=1.0):
    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.view(-1)
    targets = targets.view(-1)

    intersection = (probabilities * targets).sum()

    dice = (
        2.0 * intersection + smooth
    ) / (
        probabilities.sum() + targets.sum() + smooth
    )

    return 1.0 - dice


bce_loss = nn.BCEWithLogitsLoss()


def combined_loss(logits, targets):
    bce = bce_loss(logits, targets)
    dice = dice_loss(logits, targets)

    return bce + dice

## 7. Training U-Net

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

loss_history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for images, masks in loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(images)
        loss = combined_loss(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    mean_loss = epoch_loss / max(1, len(loader))
    loss_history.append(mean_loss)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} - "
        f"Loss: {mean_loss:.4f}"
    )

model_path = RESULT_DIR / "unet_notebook.pth"
torch.save(model.state_dict(), model_path)

print("Model saved:", model_path)

## 8. Training Loss

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(loss_history) + 1),
    loss_history,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("U-Net Training Loss")
plt.grid(True)
plt.show()

## 9. Inference

In [ ]:
model.eval()

if len(dataset) > 0:
    image, ground_truth = dataset[0]

    input_tensor = image.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(input_tensor)

    probability = torch.sigmoid(logits)[0, 0].cpu().numpy()

    prediction = (probability >= 0.5).astype(np.uint8)

    print("Input shape       :", tuple(image.shape))
    print("Output shape      :", probability.shape)
    print("Probability range :", probability.min(), "to", probability.max())
    print("Prediction values :", np.unique(prediction))
else:
    print("Không có dữ liệu để inference.")

## 10. Visualize Segmentation Result

In [ ]:
if len(dataset) > 0:
    image_np = image.permute(1, 2, 0).cpu().numpy()
    ground_truth_np = ground_truth.squeeze(0).cpu().numpy()

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(image_np)
    axes[0].set_title("Input Image")
    axes[0].axis("off")

    axes[1].imshow(ground_truth_np, cmap="gray")
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    axes[2].imshow(prediction, cmap="gray")
    axes[2].set_title("Prediction")
    axes[2].axis("off")

    axes[3].imshow(image_np)
    axes[3].imshow(prediction, alpha=0.4, cmap="Reds")
    axes[3].set_title("Overlay")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

## 11. Calculate Segmentation Metrics

In [ ]:
def calculate_metrics(ground_truth, prediction):
    ground_truth = ground_truth.astype(bool)
    prediction = prediction.astype(bool)

    tp = np.logical_and(
        ground_truth,
        prediction
    ).sum()

    fp = np.logical_and(
        ~ground_truth,
        prediction
    ).sum()

    fn = np.logical_and(
        ground_truth,
        ~prediction
    ).sum()

    dice = (2 * tp) / max(1, 2 * tp + fp + fn)
    iou = tp / max(1, tp + fp + fn)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)

    return dice, iou, precision, recall


if len(dataset) > 0:
    dice, iou, precision, recall = calculate_metrics(
        ground_truth_np,
        prediction
    )

    print(f"Dice      : {dice:.4f}")
    print(f"IoU       : {iou:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")

## 12. Segmentation Summary

U-Net có nhiệm vụ xác định pixel nào thuộc vùng tổn thương da.

Kết quả được biểu diễn dưới dạng Binary Segmentation Mask và được đánh giá bằng Dice, IoU, Precision và Recall.

In [ ]:
print("=" * 60)
print("U-NET SEGMENTATION SUMMARY")
print("=" * 60)
print("Task       : Skin Lesion Segmentation")
print("Model      : U-Net")
print("Input size :", IMAGE_SIZE)
print("Epochs     :", NUM_EPOCHS)
print("Batch size :", BATCH_SIZE)
print("Device     :", DEVICE)
print("Threshold  : 0.5")
print("Loss       : BCE + Dice Loss")
print("Metrics    : Dice / IoU / Precision / Recall")
print("=" * 60)